![alt text](./Cerny_logo_1.jpg)

# Analysis of Cerny ventilation recordings

#### Processing clinical details

This notebook imports and processes clinical data and exports it into a pickle archive.

The data processed and analysed in this Notebook were collected by the **Neonatal Emergency and Transport Service of the Peter Cerny Foundation**, Budapest, Hungary

**Author: Dr Gusztav Belteki**

- Recordings with >=5 minutes of ventilator data: **1966 cases**
- Clinical data available in **1829 cases**
- Only keep clinical data for cases where ventilation recordings (>=5 minutes) are also available: **1880 cases**.

### 1. Import the required libraries and set options

In [4]:
import IPython
import pandas as pd
import numpy as np
import scipy as sp
import matplotlib
import matplotlib.pyplot as plt

import os
import sys
import re
import pickle

from scipy import stats
from pandas import Series, DataFrame
from datetime import datetime, timedelta

%matplotlib inline
matplotlib.style.use('classic')
matplotlib.rcParams['figure.facecolor'] = 'w'

pd.set_option('display.max_rows', 250)
pd.set_option('display.max_columns', 250)
pd.set_option('mode.chained_assignment', None) 

# This is to turn off a warning message which is given when read_Excel() imports '.xlsx' files
import warnings
warnings.simplefilter("ignore")

In [5]:
print("Python version: {}".format(sys.version))
print("pandas version: {}".format(pd.__version__))
print("matplotlib version: {}".format(matplotlib.__version__))
print("NumPy version: {}".format(np.__version__))
print("SciPy version: {}".format(sp.__version__))
print("IPython version: {}".format(IPython.__version__))

Python version: 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 08:22:19) [Clang 14.0.6 ]
pandas version: 2.2.2
matplotlib version: 3.9.2
NumPy version: 1.26.4
SciPy version: 1.13.1
IPython version: 8.27.0


### 2. List and set the working directory and the directory to write out data

In [7]:
# Name of the external hard drive
DRIVE = 'GB_1'

# Directory on external drive to read the clinical from
DIR_READ = os.path.join(os.sep, 'Volumes', DRIVE, 'Ventilator_data', 'Fabian_new', 'fabian_patient_data_all_new')

# Path to project folder containing ventilation research results
PATH = os.path.join(os.sep, 'Users', 'guszti', 'Library', 'Mobile Documents', 'com~apple~CloudDocs', 
                            'Documents', 'Research', 'Ventilation')

# Folder to export the result of analysis
DIR_WRITE = os.path.join(PATH, 'ventilation_fabian_new', 'Analyses')
os.makedirs(DIR_WRITE, exist_ok = True)

# Folder on a USB stick to export data to and to import processed data exported by other Notebooks
DATA_DUMP = os.path.join(os.sep, 'Volumes', DRIVE, 'data_dump', 'fabian_new',)
os.makedirs(DATA_DUMP, exist_ok = True)

DIR_READ, DIR_WRITE, DATA_DUMP

('/Volumes/GB_1/Ventilator_data/Fabian_new/fabian_patient_data_all_new',
 '/Users/guszti/Library/Mobile Documents/com~apple~CloudDocs/Documents/Research/Ventilation/ventilation_fabian_new/Analyses',
 '/Volumes/GB_1/data_dump/fabian_new')

### 3. Import ventilation data

This is needed to know the beginning and the end of the recordings

In [9]:
tags = ['1_150', '151_300', '301_450', '451_600', '601_750', '751_900', '901_1050', '1051_1200', '1201_1350', '1351_1500',
        '1501_1650', '1651_1800', '1801_1950', '1951_2100', '2101_2250', '2251_2400',]

In [10]:
%%time

data_pars = {}
for tag in tags:
    with open(os.path.join(DATA_DUMP, f'data_pars_{tag}.pickle'), 'rb') as handle:
        dta = pickle.load(handle)
        # Only keep one parameter as placeholder to limit memory use. We will only use the index here.
        for recording in dta:
            dta[recording] = dta[recording][['FiO2']]
    data_pars.update(dta)
    
len(data_pars)

CPU times: user 12.8 s, sys: 1.64 s, total: 14.5 s
Wall time: 14.6 s


1966

Shift the time stamp of ventilator recordings recorded with incorrect time stamps

Az eltérés valóban az AT000110-estől kezdődik és az AT000216-ig tart. Sajnos volt közben egy téli/nyári váltás is. 
Március 28 után a plusz 9 órával kell korrigálni, előtte 10 órával. 

In [12]:
for case in data_pars:
    if 110 <= int(case[2:].lstrip('0')) < 195:
        data_pars[case].index = data_pars[case].index.shift(10, freq='H')
        
    elif 195 <= int(case[2:].lstrip('0')) <= 216:
        data_pars[case].index = data_pars[case].index.shift(9, freq='H')
len(data_pars)

1966

### 4. Import clinical data

In [14]:
# import text files in a dictionary
clin_dict = {}
for fname in os.listdir(DIR_READ):
    if not fname.startswith('.'): # disregard hidden files
        fhandle = open(os.path.join(DIR_READ, fname), 'r', encoding = 'cp1252', errors='replace')
        clin_dict[fname[:-4]] = fhandle.read() # use the filenames without the .txt extension as keys
        fhandle.close()

In [15]:
# split the clinical data into a list
for key in sorted(clin_dict.keys()):
    clin_dict[key] = clin_dict[key].split('\n')[:-1]

In [16]:
# Create an inner dictionary for the different clinical data
for key, value in sorted(clin_dict.items()):
    temp_dict = {}
    for item in value:
        td_key, *td_value = item.split(':')
        td_key = td_key.strip()
        temp_dict[td_key] = ''.join(td_value)[1:]
    clin_dict[key] = temp_dict

In [17]:
# Create a DataFrame from the dictionary of dictionaries
clin_df = DataFrame(clin_dict).T
clin_df.index.name = 'Recording_ID'
clin_df.sort_index(inplace = True)
# Drop column containing confidential data (name)
clin_df = clin_df.drop('Name', axis=1)
len(clin_df)

1937

### 5. Drop cases which have no clinical data

In [19]:
clin_df.dropna(axis = 0, how = 'all', inplace = True)
len(clin_df)

1937

In [20]:
writer = pd.ExcelWriter(os.path.join(DIR_WRITE, 'clinical_data_all_new_unfiltered.xlsx'))
clin_df.to_excel(writer, 'clin_df')
writer.close()

### 6. Drop cases for which there is no ventilation data

Ventilation recordings may have been excluded because they were too short (<5 minutes in total) or aberrant

In [22]:
combined = sorted(set(list(clin_df.index)) & set(data_pars.keys()))
clin_df = clin_df.loc[combined]
len(clin_df)

1880

### 7. Clean up clinical dataframe

In [24]:
# Change order of columns and create English names
clin_df = clin_df[['Esetlap id', 'Date of Birth', 'Gestation Age', 'Birth Weight', 'Actual Weight', 'Pathology', 'Start', 'End']]
clin_df.columns = ['Case ID', 'Date of Birth', 'Gestational Age', 'Birth Weight', 'Actual Weight', 'Pathology', 'Start', 'End']

In [25]:
clin_df.loc[:, 'Gestational Age'] = clin_df['Gestational Age'].map(lambda x: int(x[:2]))
clin_df.loc[:, 'Birth Weight'] = clin_df['Birth Weight'].map(lambda x: int(x[:-6]))
clin_df.loc[:, 'Actual Weight'] = clin_df['Actual Weight'].str.strip(' grams')

#### Start and end of ventilation data
This shows the time points when ventilator was turned on and off. At the beginning and the end of the recordings the baby was usually not attached to the ventilator. The ventilator recordings have been manually inspected and have been trimmed accordingly.

In [27]:
starts = {}; ends = {}
for rec in sorted(clin_df.index):
    try:
        starts[rec] = data_pars[rec].index[0]
    except KeyError:
        continue
        
    try:
        ends[rec] = data_pars[rec].index[-1]
    except KeyError:
        continue
        
start_end = DataFrame([starts, ends]).T
start_end.columns = ['Recording start', 'Recording end']

In [28]:
clin_df = pd.concat([clin_df, start_end], axis = 1, join = 'outer')
clin_df['Date of Birth'] = clin_df['Date of Birth'].map(lambda x: pd.to_datetime(x))
# The separator is ';' before AT001260, ',' after that
clin_df['Pathology'] = clin_df['Pathology'].map(lambda x: re.split('[,;]', x))
clin_df['Duration'] = clin_df['Recording end'] - clin_df['Recording start']
clin_df['Postnatal Age']   = clin_df['Recording end'] - clin_df['Date of Birth']

In [30]:
clin_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1880 entries, AT000005 to AT002335
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype          
---  ------           --------------  -----          
 0   Case ID          1880 non-null   object         
 1   Date of Birth    1880 non-null   datetime64[ns] 
 2   Gestational Age  1880 non-null   object         
 3   Birth Weight     1880 non-null   object         
 4   Actual Weight    1880 non-null   object         
 5   Pathology        1880 non-null   object         
 6   Start            1880 non-null   object         
 7   End              1880 non-null   object         
 8   Recording start  1880 non-null   datetime64[ns] 
 9   Recording end    1880 non-null   datetime64[ns] 
 10  Duration         1880 non-null   timedelta64[ns]
 11  Postnatal Age    1880 non-null   timedelta64[ns]
dtypes: datetime64[ns](3), object(7), timedelta64[ns](2)
memory usage: 190.9+ KB


In [50]:
clin_df['Gestational Age'] = pd.to_timedelta((clin_df['Gestational Age']), unit='W', errors='raise')
clin_df['Corrected gestational Age'] = pd.to_timedelta((clin_df['Gestational Age']), unit='D', errors='raise') + clin_df['Postnatal Age']
clin_df['Gestational Age (weeks)'] = clin_df['Gestational Age'].apply(lambda x: x.total_seconds() / (60 * 60 * 24 * 7))
clin_df['Corrected gestational Age (weeks)'] = \
    clin_df['Corrected gestational Age'].apply(lambda x: round(x.total_seconds() / (60 * 60 * 24 * 7), 1))

In [52]:
actual_weight = []
for i in range(len(clin_df)):
    if clin_df.iloc[i]['Actual Weight'] != '':
        actual_weight.append(int(clin_df.iloc[i]['Actual Weight']))
    #If actual weight is not available and postnatal age is <7 days, use birth weight, otherwise put np.nan
    elif clin_df.iloc[i]['Postnatal Age'] <= pd.to_timedelta('7D'):
        actual_weight.append(clin_df.iloc[i]['Birth Weight'])
    else:
        actual_weight.append(np.nan)

clin_df['Weight'] = actual_weight

In [54]:
clin_df.sort_index(axis = 1).head(2)

,Actual Weight,Birth Weight,Case ID,Corrected gestational Age,Corrected gestational Age (weeks),Date of Birth,Duration,End,Gestational Age,Gestational Age (weeks),Pathology,Postnatal Age,Recording end,Recording start,Start,Weight
AT000005,,1150,55871,203 days 03:53:22,29.0,2020-10-21 08:31:00,0 days 02:12:55,"Wed, 21 Oct 2020 122422 +0200",203 days,29.0,"[Neonat praemat gr s 29 (P07.3) , RDS (P22) ...",0 days 03:53:22,2020-10-21 12:24:22,2020-10-21 10:11:27,"Wed, 21 Oct 2020 101127 +0200",1150.0
AT000006,940,820,55875,212 days 23:53:36,30.4,2020-09-27 15:45:00,0 days 00:41:31,"Wed, 21 Oct 2020 153836 +0200",189 days,27.0,"[Neonat immat gr s 27 (P07.2) , RDS (P22) ,...",23 days 23:53:36,2020-10-21 15:38:36,2020-10-21 14:57:05,"Wed, 21 Oct 2020 145705 +0200",940.0


### 8. Exploratory analysis on clinical details

In [57]:
clin_df.describe()

,Date of Birth,Gestational Age,Recording start,Recording end,Duration,Postnatal Age,Corrected gestational Age,Gestational Age (weeks),Corrected gestational Age (weeks),Weight
count,1880,1880,1880,1880,1880,1880,1880,1880.000000,1880.000000,1842.000000
mean,2023-08-09 23:05:35.329787392,247 days 19:28:05.106382980,2023-08-22 15:08:39.809042688,2023-08-22 16:22:50.887765760,0 days 01:14:11.078723404,12 days 17:17:15.557978723,260 days 12:45:20.664361704,35.401596,37.208617,2752.793160
min,2020-09-27 15:45:00,154 days 00:00:00,2020-10-21 10:11:27,2020-10-21 12:24:22,0 days 00:05:22,0 days 00:04:33,154 days 02:58:33,22.000000,22.000000,280.000000
25%,2021-12-15 03:03:30,231 days 00:00:00,2021-12-25 06:19:34.750000128,2021-12-25 07:40:38,0 days 00:43:30,0 days 03:14:40.750000,238 days 04:47:08.750000,33.000000,34.000000,2050.000000
50%,2023-06-09 14:54:00,259 days 00:00:00,2023-06-20 09:12:43,2023-06-20 09:27:51,0 days 01:05:57,0 days 06:53:11.500000,261 days 11:46:34,37.000000,37.350000,2860.000000
75%,2025-03-26 01:31:45,273 days 00:00:00,2025-04-01 01:47:59.500000,2025-04-01 04:13:56,0 days 01:36:18.500000,4 days 12:21:54.250000,280 days 01:42:37.500000,39.000000,40.000000,3480.000000
max,2026-07-02 15:14:00,392 days 00:00:00,2026-07-03 12:56:59,2026-07-03 13:56:57,0 days 06:31:10,544 days 06:01:35,817 days 06:01:35,56.000000,116.800000,7000.000000
std,NaN,32 days 17:00:30.836367791,NaN,NaN,0 days 00:42:11.324139470,41 days 04:50:43.251594434,49 days 15:50:27.334285671,4.672670,7.096800,1014.567608


#### For some recordings the age at the time of transfer is "negative"  - these need to be corrected

In [60]:
clin_df[clin_df['Postnatal Age'] < pd.to_timedelta(0)].sort_values('Postnatal Age')

,Case ID,Date of Birth,Gestational Age,Birth Weight,Actual Weight,Pathology,Start,End,Recording start,Recording end,Duration,Postnatal Age,Corrected gestational Age,Gestational Age (weeks),Corrected gestational Age (weeks),Weight


#### For some recordings the duration of the recording is "negative"  - these need to be corrected

In [63]:
clin_df[clin_df['Duration'] < pd.to_timedelta(0)]

,Case ID,Date of Birth,Gestational Age,Birth Weight,Actual Weight,Pathology,Start,End,Recording start,Recording end,Duration,Postnatal Age,Corrected gestational Age,Gestational Age (weeks),Corrected gestational Age (weeks),Weight


#### Duration longer than 6 hours - this would be unusual for neonatal transport

In [66]:
clin_df[clin_df['Duration'] > pd.to_timedelta('6H')]

,Case ID,Date of Birth,Gestational Age,Birth Weight,Actual Weight,Pathology,Start,End,Recording start,Recording end,Duration,Postnatal Age,Corrected gestational Age,Gestational Age (weeks),Corrected gestational Age (weeks),Weight
AT000842,61060,2022-08-28 01:55:00,238 days,2350,,"[Neonat praemat gr s 34 (P07.3) , RDS (P22) ...","Sun, 28 Aug 2022 084958 +0200","Sun, 28 Aug 2022 152108 +0200",2022-08-28 08:49:58,2022-08-28 15:21:08,0 days 06:31:10,0 days 13:26:08,238 days 13:26:08,34.0,34.1,2350.0
AT001272,65673,2024-04-10 00:00:00,287 days,3620,3500,"[_Meconium aspiratio (P2400)_, PerzisztÃ¡lÃ³ ...","Sat, 13 Apr 2024 210219 +0200","Sat, 13 Apr 2024 221249 +0200",2024-04-13 21:02:19,2024-04-14 03:20:38,0 days 06:18:19,4 days 03:20:38,291 days 03:20:38,41.0,41.6,3500.0


#### Babies was at less than 23 weeks gestation

In [69]:
clin_df[clin_df['Gestational Age (weeks)'] < 23]

,Case ID,Date of Birth,Gestational Age,Birth Weight,Actual Weight,Pathology,Start,End,Recording start,Recording end,Duration,Postnatal Age,Corrected gestational Age,Gestational Age (weeks),Corrected gestational Age (weeks),Weight
AT000017,56003,2020-11-10 16:00:00,154 days,520,,"[Neonat immat gr s 22 (P07.2) , Hypothermia ...","Tue, 10 Nov 2020 175907 +0100","Tue, 10 Nov 2020 193447 +0100",2020-11-10 17:59:07,2020-11-10 19:34:47,0 days 01:35:40,0 days 03:34:47,154 days 03:34:47,22.0,22.0,520.0
AT001235,63907,2023-08-30 12:41:00,154 days,280,,"[Neonat immat gr s 22 (P07.2) , RDS (P22) ,...","Wed, 30 Aug 2023 141835 +0200","Wed, 30 Aug 2023 153933 +0200",2023-08-30 14:18:35,2023-08-30 15:39:33,0 days 01:20:58,0 days 02:58:33,154 days 02:58:33,22.0,22.0,280.0
AT001758,68634,2025-04-23 00:13:00,154 days,400,400,"[ExtrÃ©m Ã©retlensÃ©g (P0720), _Az ÃºjszÃ¼lÃ¶...","Wed, 23 Apr 2025 013212 +0200","Wed, 23 Apr 2025 035233 +0200",2025-04-23 01:32:12,2025-04-23 03:52:33,0 days 02:20:21,0 days 03:39:33,154 days 03:39:33,22.0,22.0,400.0
AT001789,68852,2025-05-22 10:10:00,154 days,470,470,"[_ExtrÃ©m Ã©retlensÃ©g, 23 betÃ¶ltÃ¶tt hÃ©tnÃ...","Thu, 22 May 2025 141959 +0200","Thu, 22 May 2025 165000 +0200",2025-05-22 14:19:59,2025-05-22 16:50:00,0 days 02:30:01,0 days 06:40:00,154 days 06:40:00,22.0,22.0,470.0
AT002142,70982,2025-10-27 00:00:00,154 days,490,1650,"[_ExtrÃ©m Ã©retlensÃ©g (P0720)_, KoraszÃ¼lÃ¶t...","Wed, 04 Feb 2026 104307 +0100","Wed, 04 Feb 2026 125208 +0100",2026-02-04 10:43:07,2026-02-04 12:52:08,0 days 02:09:01,100 days 12:52:08,254 days 12:52:08,22.0,36.4,1650.0


#### Babies born with less than 500 g birth weight

In [72]:
clin_df[clin_df['Birth Weight'] < 500]

,Case ID,Date of Birth,Gestational Age,Birth Weight,Actual Weight,Pathology,Start,End,Recording start,Recording end,Duration,Postnatal Age,Corrected gestational Age,Gestational Age (weeks),Corrected gestational Age (weeks),Weight
AT000129,56646,2021-01-23 00:00:00,203 days,460,620,"[Neonat praemat gr s 29 (P07.3) , Dysmaturit...","Thu, 11 Feb 2021 114454 +0100","Thu, 11 Feb 2021 123425 +0100",2021-02-11 12:44:54,2021-02-11 13:34:25,0 days 00:49:31,19 days 13:34:25,222 days 13:34:25,29.0,31.8,620.0
AT000357,58015,2021-08-03 00:00:00,280 days,320,,"[Neonat mat gr s 40 (P96.4) , Asphyxia perin...","Wed, 04 Aug 2021 001908 +0200","Wed, 04 Aug 2021 010710 +0200",2021-08-04 00:19:08,2021-08-04 01:07:10,0 days 00:48:02,1 days 01:07:10,281 days 01:07:10,40.0,40.1,320.0
AT000686,59909,2022-04-16 05:07:00,168 days,450,,"[Neonat immat gr s 24 (P07.2) , RDS (P22) ,...","Sat, 16 Apr 2022 062815 +0200","Sat, 16 Apr 2022 090723 +0200",2022-04-16 06:28:15,2022-04-16 09:07:23,0 days 02:39:08,0 days 04:00:23,168 days 04:00:23,24.0,24.0,450.0
AT000916,62119,2022-12-17 12:19:00,182 days,490,490,"[Neonat immat gr s 26 (P07.2) , RDS (P22) ,...","Tue, 27 Dec 2022 155336 +0100","Tue, 27 Dec 2022 162456 +0100",2022-12-27 15:53:36,2022-12-27 16:24:56,0 days 00:31:20,10 days 04:05:56,192 days 04:05:56,26.0,27.5,490.0
AT001123,63199,2023-03-24 00:00:00,168 days,410,1340,"[Neonat immat gr s 24 (P07.2) , BPD (P27.1) ...","Wed, 31 May 2023 105159 +0200","Wed, 31 May 2023 113043 +0200",2023-05-31 10:51:59,2023-05-31 11:30:43,0 days 00:38:44,68 days 11:30:43,236 days 11:30:43,24.0,33.8,1340.0
AT001126,63217,2023-03-24 00:00:00,168 days,410,1380,"[Sepsis (P36.9) , Légzészavar (P22.8) , Ne...","Fri, 02 Jun 2023 104943 +0200","Fri, 02 Jun 2023 113505 +0200",2023-06-02 10:49:43,2023-06-02 11:35:05,0 days 00:45:22,70 days 11:35:05,238 days 11:35:05,24.0,34.1,1380.0
AT001235,63907,2023-08-30 12:41:00,154 days,280,,"[Neonat immat gr s 22 (P07.2) , RDS (P22) ,...","Wed, 30 Aug 2023 141835 +0200","Wed, 30 Aug 2023 153933 +0200",2023-08-30 14:18:35,2023-08-30 15:39:33,0 days 01:20:58,0 days 02:58:33,154 days 02:58:33,22.0,22.0,280.0
AT001521,67134,2024-10-02 00:00:00,182 days,400,390,"[_ÃšjszÃ¼lÃ¶ttkori tÃ¼dÅ‘vÃ©rzÃ©s, k.m.n. (P2...","Fri, 11 Oct 2024 223939 +0200","Fri, 11 Oct 2024 233927 +0200",2024-10-11 22:39:39,2024-10-11 23:39:27,0 days 00:59:48,9 days 23:39:27,191 days 23:39:27,26.0,27.4,390.0
AT001758,68634,2025-04-23 00:13:00,154 days,400,400,"[ExtrÃ©m Ã©retlensÃ©g (P0720), _Az ÃºjszÃ¼lÃ¶...","Wed, 23 Apr 2025 013212 +0200","Wed, 23 Apr 2025 035233 +0200",2025-04-23 01:32:12,2025-04-23 03:52:33,0 days 02:20:21,0 days 03:39:33,154 days 03:39:33,22.0,22.0,400.0
AT001789,68852,2025-05-22 10:10:00,154 days,470,470,"[_ExtrÃ©m Ã©retlensÃ©g, 23 betÃ¶ltÃ¶tt hÃ©tnÃ...","Thu, 22 May 2025 141959 +0200","Thu, 22 May 2025 165000 +0200",2025-05-22 14:19:59,2025-05-22 16:50:00,0 days 02:30:01,0 days 06:40:00,154 days 06:40:00,22.0,22.0,470.0



#### Babies transferred with the postnatal age of > 46 weeks. We need to discuss whether to include them in the data analysis.

In [75]:
len(clin_df[clin_df['Corrected gestational Age (weeks)'] > 46])

77

### 9. Identify ICD codes

In [78]:
icd = re.compile(r'\(\S\d+\.?\d*\)')

In [80]:
def icd_finder(lst):
    icd_list = []
    for item in lst:
        if icd.findall(item):
            icd_list.append(icd.findall(item)[0])   
    return icd_list

In [82]:
def icd_cleanup(lst):
    icd_list = []
    for item in lst:
        if '.' in item:
            new_item = item[0 : item.index('.')] + item[item.index('.') + 1 : ]
        else:
            new_item = item
        icd_list.append(new_item[1:-1])
    return icd_list

In [84]:
def icd_cleanup_2(lst):
    # Some BNO codes have 4 digits and end with zero. These are the same codes as the same ones with 3 digits (without zero at the end)
    icd_list = []
    for item in lst:
        if len(item) == 5 and item.endswith('0'):
            icd_list.append(item[:-1])
        else:
            icd_list.append(item)
    icd_lst = sorted(set(icd_list))
    return icd_list

In [86]:
clin_df['ICD'] = clin_df['Pathology'].apply(icd_finder)
clin_df['ICD'] = clin_df['ICD'].apply(icd_cleanup)
clin_df['ICD'] = clin_df['ICD'].apply(icd_cleanup_2)
clin_df['ICD'];

### 10. Identify all icd codes

In [89]:
icd_all = []
for _, item in clin_df.iterrows():
    icd_all.extend(item['ICD']) 
icd_all = sorted(set(icd_all))
len(icd_all)

376

In [91]:
icd_all_frme = DataFrame(icd_all)
icd_all_frme.columns = ['code']
icd_all_frme.sort_values(by = 'code', inplace = True)
len(icd_all_frme)

376

In [93]:
writer = pd.ExcelWriter(os.path.join(DIR_WRITE, 'icd_codes.xlsx'))
icd_all_frme.to_excel(writer, 'icd_codes')
writer.close()

### 11. Import the file that has been manually curated from the exported `icd_codes.xlsx` files to contain all relevant diagnosis previously identified

In [96]:
icd_codes = pd.read_excel(os.path.join(PATH, 'ventilation_fabian_new', 'icd_codes_curated_new.xlsx'), usecols = [0,1], index_col = 0)

Identify new codes not in the dataset so far

In [99]:
new_diagnoses = sorted(set(icd_all_frme['code']) - set(icd_codes.index))
new_diagnoses

['A50', 'G918', 'O809', 'P0725', 'Q600', 'S060']

At this point the `icd_codes_curated_new.xlsx` file needs to be manually curated with these entries.

### 12. Import the now curated `icd_codes.xlsx` files to contain now all relevant diagnosis including new ones

In [103]:
icd_codes = pd.read_excel(os.path.join(PATH, 'ventilation_fabian_new', 'icd_codes_curated_new.xlsx'),
    usecols = [0,1], index_col = 0)

In [105]:
# Now there are no new codes
sorted(set(icd_all_frme['code']) - set(icd_codes.index))

[]

### 13. Create Pathology column with English names

In [108]:
icd_dictionary = dict(zip(icd_codes.index, icd_codes['name']))
icd_dictionary;

In [110]:
def icd_replace(lst):
    icd_list = []
    for item in lst:
        new_item = icd_dictionary[item]
        icd_list.append(new_item)
    return icd_list

In [112]:
clin_df['Pathology_English'] = clin_df['ICD'].apply(icd_replace)

In [114]:
clin_df.tail()

,Case ID,Date of Birth,Gestational Age,Birth Weight,Actual Weight,Pathology,Start,End,Recording start,Recording end,Duration,Postnatal Age,Corrected gestational Age,Gestational Age (weeks),Corrected gestational Age (weeks),Weight,ICD,Pathology_English
AT002330,72310,2026-06-24 11:06:00,259 days,3280,3170,"[_KÃ¶zÃ¶s artÃ©riÃ¡s tÃ¶rzs (Q2000)_, PangÃ¡s...","Thu, 02 Jul 2026 110540 +0200","Thu, 02 Jul 2026 111424 +0200",2026-07-02 11:05:40,2026-07-02 11:14:24,0 days 00:08:44,8 days 00:08:24,267 days 00:08:24,37.0,38.1,3170.0,"[Q200, I500, P073, Q211, Q381]","[Common arterial trunk, Congestive heart failu..."
AT002332,72316,2026-07-02 15:14:00,273 days,3250,3250,"[_ÃšjszÃ¼lÃ¶ttkori polycythaemia (P6110)_, Az...","Thu, 02 Jul 2026 210735 +0200","Thu, 02 Jul 2026 214405 +0200",2026-07-02 21:07:35,2026-07-02 21:44:05,0 days 00:36:30,0 days 06:30:05,273 days 06:30:05,39.0,39.0,3250.0,"[P611, P228, P964]","[Polycythemia neonatorum, Other respiratory di..."
AT002333,72323,2026-06-27 13:30:00,273 days,2980,3090,"[A terhessÃ©g befejezÅ‘dÃ©se, magzat Ã©s Ãºjs...","Fri, 03 Jul 2026 095000 +0200","Fri, 03 Jul 2026 103808 +0200",2026-07-03 09:50:00,2026-07-03 10:38:08,0 days 00:48:08,5 days 21:08:08,278 days 21:08:08,39.0,39.8,3090.0,"[P964, Q203, Q262, Q221]","[Term newborn, Transposition of the great arte..."
AT002334,72324,2026-06-24 11:06:00,259 days,3280,3200,"[A terhessÃ©g befejezÅ‘dÃ©se, magzat Ã©s Ãºjs...","Fri, 03 Jul 2026 111516 +0200","Fri, 03 Jul 2026 115900 +0200",2026-07-03 11:15:16,2026-07-03 11:59:00,0 days 00:43:44,9 days 00:53:00,268 days 00:53:00,37.0,38.3,3200.0,"[P964, Q200]","[Term newborn, Common arterial trunk]"
AT002335,72327,2026-06-20 00:00:00,280 days,3380,3380,"[A terhessÃ©g befejezÅ‘dÃ©se, magzat Ã©s Ãºjs...","Fri, 03 Jul 2026 125659 +0200","Fri, 03 Jul 2026 135657 +0200",2026-07-03 12:56:59,2026-07-03 13:56:57,0 days 00:59:58,13 days 13:56:57,293 days 13:56:57,40.0,41.9,3380.0,"[P964, S060]","[Term newborn, Concussion without loss of cons..."


### 14. Final cleanup of the DataFrame

In [117]:
clin_df.columns

Index(['Case ID', 'Date of Birth', 'Gestational Age', 'Birth Weight',
       'Actual Weight', 'Pathology', 'Start', 'End', 'Recording start',
       'Recording end', 'Duration', 'Postnatal Age',
       'Corrected gestational Age', 'Gestational Age (weeks)',
       'Corrected gestational Age (weeks)', 'Weight', 'ICD',
       'Pathology_English'],
      dtype='object')

In [119]:
column_list = ['Case ID', 'Date of Birth', 'Gestational Age (weeks)', 'Birth Weight', 'Postnatal Age', 
               'Corrected gestational Age (weeks)',  'Weight','ICD', 'Pathology_English', 'Recording start', 
               'Recording end', 'Duration',] 
clin_df = clin_df[column_list]

In [121]:
clin_df.tail()

,Case ID,Date of Birth,Gestational Age (weeks),Birth Weight,Postnatal Age,Corrected gestational Age (weeks),Weight,ICD,Pathology_English,Recording start,Recording end,Duration
AT002330,72310,2026-06-24 11:06:00,37.0,3280,8 days 00:08:24,38.1,3170.0,"[Q200, I500, P073, Q211, Q381]","[Common arterial trunk, Congestive heart failu...",2026-07-02 11:05:40,2026-07-02 11:14:24,0 days 00:08:44
AT002332,72316,2026-07-02 15:14:00,39.0,3250,0 days 06:30:05,39.0,3250.0,"[P611, P228, P964]","[Polycythemia neonatorum, Other respiratory di...",2026-07-02 21:07:35,2026-07-02 21:44:05,0 days 00:36:30
AT002333,72323,2026-06-27 13:30:00,39.0,2980,5 days 21:08:08,39.8,3090.0,"[P964, Q203, Q262, Q221]","[Term newborn, Transposition of the great arte...",2026-07-03 09:50:00,2026-07-03 10:38:08,0 days 00:48:08
AT002334,72324,2026-06-24 11:06:00,37.0,3280,9 days 00:53:00,38.3,3200.0,"[P964, Q200]","[Term newborn, Common arterial trunk]",2026-07-03 11:15:16,2026-07-03 11:59:00,0 days 00:43:44
AT002335,72327,2026-06-20 00:00:00,40.0,3380,13 days 13:56:57,41.9,3380.0,"[P964, S060]","[Term newborn, Concussion without loss of cons...",2026-07-03 12:56:59,2026-07-03 13:56:57,0 days 00:59:58


### 15. Statistics on clinical data

In [124]:
clinical_stats = round(clin_df.describe(percentiles = [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]), 1)
clinical_stats

,Date of Birth,Gestational Age (weeks),Postnatal Age,Corrected gestational Age (weeks),Weight,Recording start,Recording end,Duration
count,1880,1880.0,1880,1880.0,1842.0,1880,1880,1880
mean,2023-08-09 23:05:35.329787392,35.4,12 days 17:17:15.557978723,37.2,2752.8,2023-08-22 15:08:39.809042688,2023-08-22 16:22:50.887765760,0 days 01:14:11.078723404
min,2020-09-27 15:45:00,22.0,0 days 00:04:33,22.0,280.0,2020-10-21 10:11:27,2020-10-21 12:24:22,0 days 00:05:22
1%,2020-11-16 04:33:27.600000,24.0,0 days 01:06:52.160000,24.8,600.0,2020-11-17 17:54:24.049999872,2020-11-17 19:19:40.920000,0 days 00:10:23.950000
5%,2021-01-27 23:09:24,25.0,0 days 01:43:33.900000,28.0,990.0,2021-01-31 21:49:03.550000128,2021-01-31 22:47:27.800000,0 days 00:24:41.800000
25%,2021-12-15 03:03:30,33.0,0 days 03:14:40.750000,34.0,2050.0,2021-12-25 06:19:34.750000128,2021-12-25 07:40:38,0 days 00:43:30
50%,2023-06-09 14:54:00,37.0,0 days 06:53:11.500000,37.3,2860.0,2023-06-20 09:12:43,2023-06-20 09:27:51,0 days 01:05:57
75%,2025-03-26 01:31:45,39.0,4 days 12:21:54.250000,40.0,3480.0,2025-04-01 01:47:59.500000,2025-04-01 04:13:56,0 days 01:36:18.500000
95%,2026-03-24 19:53:42,40.0,68 days 18:47:58.499999999,45.0,4300.0,2026-04-17 17:55:11.349999872,2026-04-17 18:46:37.249999872,0 days 02:34:04.500000
99%,2026-06-16 20:10:09.600000,41.0,134 days 23:29:46.800000008,56.2,5008.8,2026-06-20 06:31:55.500000,2026-06-20 08:46:29.990000128,0 days 03:26:18.680000


### 16. Export clinical information data in tables and individual text files

##### Create sub-directories for each case if it does not yet exist

In [128]:
# Images and raw data will be written on an external hard drive
os.makedirs(os.path.join(DATA_DUMP, 'fabian_cases_new'), exist_ok = True)

for case in sorted(clin_df.index): 
    os.makedirs(os.path.join(DATA_DUMP, 'fabian_cases_new', case), exist_ok = True)

##### Export clinical data about individual cases as text files

In [131]:
# Clinical info about all recordings which clinical data are available and are over 15 minutes long

for case in sorted(clin_df.index):
    
    fileout = open(os.path.join(DATA_DUMP, 'fabian_cases_new', case, f'{case}_clin_info.txt'), 'w')
    
    fileout.write('Case ID:             %-50s\n' % case)
    fileout.write('Start:               %-50s\n' % datetime.strftime(clin_df.loc[case]['Recording start'], '%d/%m/%Y %H:%M:%S', ))
    fileout.write('End:                 %-50s\n' % datetime.strftime(clin_df.loc[case]['Recording end'], '%d/%m/%Y %H:%M:%S', ))
    fileout.write('Duration:            %-50s\n' % clin_df.loc[case]['Duration'])
    fileout.write('Gestational age:     %-50s\n' % clin_df.loc[case]['Gestational Age (weeks)'])
    fileout.write('Postmenstrual age:   %-50s\n' % clin_df.loc[case]['Corrected gestational Age (weeks)'])
    fileout.write('Birth Weight:        %-50s\n' % clin_df.loc[case]['Birth Weight'])
    fileout.write('Weight:              %-50s\n' % clin_df.loc[case]['Weight'])
    fileout.write('ICD:                 %-50s\n' % ', '.join(clin_df.loc[case]['ICD']))
    fileout.write('Diagnoses:           %-50s\n' % ', '.join(clin_df.loc[case]['Pathology_English']))
    
    fileout.close()

##### Export clinical information as an Excel sheet

In [133]:
writer = pd.ExcelWriter(os.path.join(DIR_WRITE, 'clinical_data_all_new.xlsx'))
clin_df.to_excel(writer, 'clin_df')
writer.close()

### 17. Export statistics on clinical data

In [135]:
writer = pd.ExcelWriter(os.path.join(DIR_WRITE, 'clinical_stats_new.xlsx'))
clinical_stats.to_excel(writer, 'stats')
writer.close()

### 18. Export processed data as pickle files

In [140]:
with open(os.path.join(DATA_DUMP, 'clin_df_new.pickle'), 'wb') as handle:
    pickle.dump(clin_df, handle, protocol=pickle.HIGHEST_PROTOCOL)

### 19. Create patient lists for various disease groups

#### A. RDS

In [144]:
RDS_dg = {'P22', 'P220'}

In [146]:
RDS_cases = []
for case, dgs in clin_df['ICD'].items():
    if set(dgs).intersection(RDS_dg):
        RDS_cases.append(case)

In [148]:
clin_df_RDS = clin_df.loc[RDS_cases]
clin_df_RDS;

In [150]:
len(clin_df_RDS)

470

#### B. HIE

In [153]:
HIE_dg = ['P219', 'Z518', 'Z548',]  

In [155]:
HIE_cases = []
for case, dgs in clin_df['ICD'].items():
    if set(dgs).intersection(HIE_dg):
        HIE_cases.append(case)

In [157]:
clin_df_HIE = clin_df.loc[HIE_cases]
clin_df_HIE;

In [159]:
len(clin_df_HIE)

285

#### C. Meconium aspiration

In [162]:
MAS_dg = ['P240',]

In [164]:
MAS_cases = []
for case, dgs in clin_df['ICD'].items():
    if set(dgs).intersection(MAS_dg):
        MAS_cases.append(case)

In [166]:
clin_df_MAS = clin_df.loc[MAS_cases]
clin_df_MAS;

In [168]:
len(clin_df_MAS)

165

#### D. PPHN

In [171]:
PPHN_dg = ['P293', ]

In [173]:
PPHN_cases = []
for case, dgs in clin_df['ICD'].items():
    if set(dgs).intersection(PPHN_dg):
        PPHN_cases.append(case)

In [175]:
clin_df_PPHN = clin_df.loc[PPHN_cases]
clin_df_PPHN;

In [177]:
len(clin_df_PPHN)

94

#### E. Congenital diaphragmatic hernia

In [180]:
CDH_dg = ['Q790', 'Q791', 'K449']

In [182]:
CDH_cases = []
for case, dgs in clin_df['ICD'].items():
    if set(dgs).intersection(CDH_dg):
        CDH_cases.append(case)

In [184]:
clin_df_CDH = clin_df.loc[CDH_cases]
clin_df_CDH;

In [186]:
len(clin_df_CDH)

28

#### F. Necrotizing enterocolitis (NEC)

In [189]:
NEC_dg = ['P77',]

In [191]:
NEC_cases = []
for case, dgs in clin_df['ICD'].items():
    if set(dgs).intersection(NEC_dg):
        NEC_cases.append(case)

In [193]:
clin_df_NEC = clin_df.loc[NEC_cases]
clin_df_NEC;

In [195]:
len(clin_df_NEC)

33

#### G. Surgical cases (except NEC, CDH and cardiac)

In [198]:
surgical_dg = ['K409', 'K449', 'K562', 'K566', 'K631', 'K921', 'K9210',  'Q019',
               'Q059',  'Q300', 'Q319', 'Q321', 'Q330',  'Q391', 'Q392', 'Q410', 'Q423' , 'Q428', 
               'Q431', 'Q433', 'Q438', 'Q4380', 'Q445', 'Q512', 'Q549', 'Q556', 'Q620', 'Q621', 'Q639',
               'Q641', 'Q642', 'Q791', 'Q792', 'Q793', 'Q848', 'R1000', 'Z432', 'Z433', 'Z921', 'Z930', 
               'Z931', 'Z9320', 'Z933' ]

In [200]:
surgical_cases = []
for case, dgs in clin_df['ICD'].items():
    if set(dgs).intersection(surgical_dg):
        surgical_cases.append(case)

In [202]:
clin_df_surgical = clin_df.loc[surgical_cases]
clin_df_surgical;

In [204]:
len(clin_df_surgical)

118

#### H. Cardiac cases (except PFO/ASD2 and PDA)

Exclude:
- Q211:	Atrial septal defect
- Q250	Patent ductus arteriosus

In [207]:
cardiac_dg = ['I270', 'I30', 'I340', 'I342', 'I421', 'I429', 'I442', 'I443', 'I471', 'I472', 'I479', 'I500', 'I509', 'I514', 'I517',
              'Q200', 'Q201', 'Q203', 'Q204', 'Q205', 'Q208', 'Q209', 'Q210', 'Q212', 'Q213', 'Q220', 'Q221', 'Q223', 'Q224', 'Q225', 
              'Q228', 'Q230', 'Q232', 'Q234', 'Q240', 'Q244', 'Q245', 'Q249', 'Q251', 'Q252', 'Q253', 'Q254', 'Q256', 'Q262', 'Q273', 
              'Q272', 'Q279']

In [209]:
cardiac_cases = []
for case, dgs in clin_df['ICD'].items():
    if set(dgs).intersection(cardiac_dg):
        cardiac_cases.append(case)

In [211]:
clin_df_cardiac = clin_df.loc[cardiac_cases]
clin_df_cardiac;

In [213]:
len(clin_df_cardiac)

284

### 20. Export clinical dataframes into a multisheet Excel file

In [216]:
writer = pd.ExcelWriter(os.path.join(DIR_WRITE, 'clinical_data_diseases_new.xlsx'))

clin_df.to_excel(writer, 'all')
clin_df_cardiac.to_excel(writer, 'cardiac')
clin_df_CDH.to_excel(writer, 'CDH')
clin_df_HIE.to_excel(writer, 'HIE')
clin_df_MAS.to_excel(writer, 'MAS')
clin_df_NEC.to_excel(writer, 'NEC')
clin_df_PPHN.to_excel(writer, 'PPHN')
clin_df_RDS.to_excel(writer, 'RDS')
clin_df_surgical.to_excel(writer, 'surgical')

writer.close()

### 21. Export selected clinical data as pickle archive

In [219]:
with open(os.path.join(DATA_DUMP, 'clin_df_new_cardiac.pickle'), 'wb') as handle:
    pickle.dump(clin_df_cardiac, handle, protocol=pickle.HIGHEST_PROTOCOL)
with open(os.path.join(DATA_DUMP, 'clin_df_new_CDH.pickle'), 'wb') as handle:
    pickle.dump(clin_df_CDH, handle, protocol=pickle.HIGHEST_PROTOCOL)
with open(os.path.join(DATA_DUMP, 'clin_df_new_HIE.pickle'), 'wb') as handle:
    pickle.dump(clin_df_HIE, handle, protocol=pickle.HIGHEST_PROTOCOL)
with open(os.path.join(DATA_DUMP, 'clin_df_new_MAS.pickle'), 'wb') as handle:
    pickle.dump(clin_df_MAS, handle, protocol=pickle.HIGHEST_PROTOCOL)
with open(os.path.join(DATA_DUMP, 'clin_df_new_NEC.pickle'), 'wb') as handle:
    pickle.dump(clin_df_NEC, handle, protocol=pickle.HIGHEST_PROTOCOL)  
with open(os.path.join(DATA_DUMP, 'clin_df_new_PPHN.pickle'), 'wb') as handle:
    pickle.dump(clin_df_PPHN, handle, protocol=pickle.HIGHEST_PROTOCOL)
with open(os.path.join(DATA_DUMP, 'clin_df_new_RDS.pickle'), 'wb') as handle:
    pickle.dump(clin_df_RDS, handle, protocol=pickle.HIGHEST_PROTOCOL)  
with open(os.path.join(DATA_DUMP, 'clin_df_new_surgical.pickle'), 'wb') as handle:
    pickle.dump(clin_df_surgical, handle, protocol=pickle.HIGHEST_PROTOCOL)     